  ## WhatsApp Group Chat Analyzer

**Minor Project — Python + NumPy**

- **Student Name:** MANGESH SHANKAR HATEKAR
- **Batch:** AUGUST


## Feature 1 — Chat Parser

The parser handles:
- normal WhatsApp message lines
- system messages
- `<Media omitted>`
- `This message was deleted`
- multi-line message continuation
- empty lines

The source brief specifies the WhatsApp Android format `DD/MM/YY, HH:MM - Sender: Message`.

In [ ]:
import os
import string
import numpy as np
from datetime import datetime, timedelta

# AI-assisted: used as a learning aid for structuring the notebook and debugging logic.
# The final submission should be reviewed and adapted in the student's own style.

CHAT_FILE = "hostel_bois.txt"

DATE_PREFIX_LENGTH = 8

def looks_like_chat_line(line):
    """Return True when the first 8 characters resemble DD/MM/YY."""
    if len(line) < DATE_PREFIX_LENGTH:
        return False
    prefix = line[:DATE_PREFIX_LENGTH]
    return (
        prefix[2] == "/" and prefix[5] == "/" and
        prefix[0:2].isdigit() and prefix[3:5].isdigit() and prefix[6:8].isdigit()
    )

def parse_chat_file(file_path):
    messages = []
    system_count = 0
    media_count = {}
    deleted_count = {}
    raw_message_entries = 0

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    pending = None

    def save_pending():
        nonlocal pending
        if pending is not None:
            messages.append(pending)
            pending = None

    for raw_line in lines:
        line = raw_line.rstrip("\n\r")

        if line.strip() == "":
            continue

        if not looks_like_chat_line(line):
            if pending is not None:
                pending["text"] += " " + line.strip()
            continue

        save_pending()

        try:
            timestamp_text, rest = line.split(" - ", 1)
        except ValueError:
            system_count += 1
            continue

        try:
            sender, text = rest.split(": ", 1)
        except ValueError:
            # Timestamp exists, but there is no sender/message separator.
            system_count += 1
            continue

        timestamp_text = timestamp_text.strip()
        sender = sender.strip()
        text = text.strip()
        raw_message_entries += 1

        pending = {
            "timestamp": timestamp_text,
            "sender": sender,
            "text": text
        }

    save_pending()

    # Count special message types separately.
    analyzable_messages = []
    for item in messages:
        sender = item["sender"]
        text = item["text"]

        if text == "<Media omitted>":
            media_count[sender] = media_count.get(sender, 0) + 1
        elif text == "This message was deleted":
            deleted_count[sender] = deleted_count.get(sender, 0) + 1
        else:
            analyzable_messages.append(item)

    return {
        "messages": messages,
        "analyzable_messages": analyzable_messages,
        "system_count": system_count,
        "media_count": media_count,
        "deleted_count": deleted_count,
        "raw_message_entries": raw_message_entries
    }

if not os.path.exists(CHAT_FILE):
    print("Dataset not found.")
    print("Upload hostel_bois.txt to the same Colab/session folder and run this cell again.")
else:
    parsed = parse_chat_file(CHAT_FILE)
    print("Parser loaded successfully.")
    print("Message entries:", len(parsed["messages"]))
    print("Analyzable text messages:", len(parsed["analyzable_messages"]))
    print("System messages:", parsed["system_count"])
    print("Media entries:", sum(parsed["media_count"].values()))
    print("Deleted entries:", sum(parsed["deleted_count"].values()))

Parser loaded successfully.
Message entries: 3174
Analyzable text messages: 3127
System messages: 4
Media entries: 32
Deleted entries: 15


# New Section

## Feature 2 — Group Overview

This section calculates the date range, number of active participants, total analyzed messages, and messages per participant.

In [ ]:
def parse_timestamp(timestamp_text):
    return datetime.strptime(timestamp_text, "%d/%m/%y, %H:%M")

def build_basic_stats(messages):
    people = set()
    per_person = {}
    dates = set()

    for msg in messages:
        person = msg["sender"]
        people.add(person)
        per_person[person] = per_person.get(person, 0) + 1
        dates.add(parse_timestamp(msg["timestamp"]).date())

    sorted_people = sorted(per_person.items(), key=lambda x: x[1], reverse=True)

    return people, per_person, dates, sorted_people

if "parsed" in globals() and parsed["analyzable_messages"]:
    messages = parsed["analyzable_messages"]
    people, per_person, active_dates, ranked_people = build_basic_stats(messages)

    first_dt = min(parse_timestamp(m["timestamp"]) for m in messages)
    last_dt = max(parse_timestamp(m["timestamp"]) for m in messages)
    period_days = (last_dt.date() - first_dt.date()).days + 1

    print("=" * 64)
    print("GROUPDNA REPORT — GROUP OVERVIEW")
    print("=" * 64)
    print(f"Period       : {first_dt.strftime('%d %B %Y')} to {last_dt.strftime('%d %B %Y')}")
    print(f"Analyzed msgs: {len(messages):,}")
    print(f"Participants : {len(people)}")
    print(f"Chat period  : {period_days} days")
    print()
    print("MESSAGES PER PERSON")

    for person, count in ranked_people:
        pct = count / len(messages) * 100
        bar_len = max(1, round(pct / 5))
        print(f"{person:<12} {'█' * bar_len:<20} {count:>5,} ({pct:>5.1f}%)")
else:
    print("Run Feature 1 after uploading hostel_bois.txt.")

GROUPDNA REPORT — GROUP OVERVIEW
Period       : 01 April 2024 to 30 May 2024
Analyzed msgs: 3,127
Participants : 6
Chat period  : 60 days

MESSAGES PER PERSON
Rahul        ██████                 940 ( 30.1%)
Priya        █████                  712 ( 22.8%)
Neha         ████                   624 ( 20.0%)
Aman         ███                    484 ( 15.5%)
Karan        ██                     345 ( 11.0%)
Vikas        █                       22 (  0.7%)


## Feature 3 — Most Active Day and Hour

The busiest day is the date with the highest message count. The busiest hour is the hour-of-day with the highest combined volume.

In [ ]:
def busiest_day_and_hour(messages):
    day_counts = {}
    hour_counts = {}

    for msg in messages:
        dt = parse_timestamp(msg["timestamp"])
        day_key = dt.date()
        hour_key = dt.hour

        day_counts[day_key] = day_counts.get(day_key, 0) + 1
        hour_counts[hour_key] = hour_counts.get(hour_key, 0) + 1

    busiest_day = max(day_counts, key=day_counts.get)
    busiest_hour = max(hour_counts, key=hour_counts.get)

    return (
        busiest_day,
        day_counts[busiest_day],
        busiest_hour,
        hour_counts[busiest_hour],
        day_counts,
        hour_counts
    )

if "messages" in globals() and messages:
    busiest_day, busiest_day_count, busiest_hour, busiest_hour_count, day_counts, hour_counts = busiest_day_and_hour(messages)

    print(f"Busiest day : {busiest_day.strftime('%d %B %Y')} ({busiest_day_count} messages)")
    print(f"Busiest hour: {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00 ({busiest_hour_count} messages)")

Busiest day : 04 May 2024 (74 messages)
Busiest hour: 18:00 - 19:00 (244 messages)


## Feature 4 — NumPy Activity Heatmap

A `participants × 24` NumPy matrix stores message totals by participant and hour. The printed heatmap uses four intensity levels, as required by the brief.

In [ ]:
def build_activity_matrix(messages, participant_order):
    matrix = np.zeros((len(participant_order), 24), dtype=int)
    row_index = {}

    for i, person in enumerate(participant_order):
        row_index[person] = i

    for msg in messages:
        dt = parse_timestamp(msg["timestamp"])
        matrix[row_index[msg["sender"]], dt.hour] += 1

    return matrix

def heat_symbol(value, row_max):
    if value == 0 or row_max == 0:
        return ". "
    ratio = value / row_max
    if ratio <= 0.25:
        return "░ "
    if ratio <= 0.50:
        return "▒ "
    return "█ "

if "ranked_people" in globals():
    participant_order = [name for name, _ in ranked_people]
    activity_matrix = build_activity_matrix(messages, participant_order)

    print("ACTIVITY HEATMAP (messages by hour)")
    print("       " + "".join(f"{h:02d}" + ("  " if h < 9 else " ") for h in range(24)))

    for i, person in enumerate(participant_order):
        row_max = int(np.max(activity_matrix[i]))
        rendered = "".join(heat_symbol(int(v), row_max) for v in activity_matrix[i])
        print(f"{person:<8}{rendered}")

    print()
    print("NumPy matrix shape:", activity_matrix.shape)
    print("Total matrix messages:", int(np.sum(activity_matrix)))

ACTIVITY HEATMAP (messages by hour)
       00  01  02  03  04  05  06  07  08  09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul   ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █ 
Priya   . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░ 
Neha    . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒ 
Aman    █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █ 
Karan   . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░ 
Vikas   . . . . . . . ▒ █ ▒ ▒ . ▒ █ . ▒ ▒ █ █ █ ▒ ▒ ▒ █ 

NumPy matrix shape: (6, 24)
Total matrix messages: 3127


## Feature 5 — Top Words

Words are normalized to lowercase, punctuation is stripped, and a manually defined stop-word set is used. A dictionary is used instead of `collections.Counter`.

In [ ]:
STOP_WORDS = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of", "in", "on",
    "for", "it", "this", "that", "was", "are", "be", "am", "we", "you",
    "me", "my", "your", "our", "with", "at", "from", "as", "but", "so",
    "have", "has", "had", "he", "she", "they", "them", "his", "her"
}

def tokenize(text):
    cleaned = text.lower()
    words = cleaned.split()
    tokens = []

    for word in words:
        word = word.strip(string.punctuation)
        if word and word not in STOP_WORDS:
            tokens.append(word)

    return tokens

def word_frequencies(messages):
    counts = {}

    for msg in messages:
        for word in tokenize(msg["text"]):
            counts[word] = counts.get(word, 0) + 1

    return counts

def print_word_bars(word_counts, limit=10):
    ranked = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:limit]

    print("THIS GROUP'S FAVOURITE WORDS")
    if not ranked:
        print("No words available.")
        return

    highest = ranked[0][1]
    for word, count in ranked:
        bar_size = max(1, round(count / highest * 20))
        print(f"{word:<15} {'█' * bar_size:<20} {count}")

if "messages" in globals() and messages:
    group_words = word_frequencies(messages)
    print_word_bars(group_words, 10)

THIS GROUP'S FAVOURITE WORDS
how             ████████████████████ 321
guys            ████████████████████ 318
about           █████████████████    274
hai             █████████████████    268
today           ████████████████     257
just            █████████████        208
which           █████████████        202
everyone        ████████████         187
telling         ███████████          179
up              ███████████          172


## Feature 6 — Response Speed and Silent Streaks

Response time is measured from a message sent by another person to the next message sent by the participant. Silent streaks count consecutive calendar days with zero messages.

In [ ]:
def response_times(messages, participants):
    ordered = sorted(messages, key=lambda x: parse_timestamp(x["timestamp"]))
    gaps = {person: [] for person in participants}

    for i in range(1, len(ordered)):
        previous = ordered[i - 1]
        current = ordered[i]

        if previous["sender"] != current["sender"]:
            previous_dt = parse_timestamp(previous["timestamp"])
            current_dt = parse_timestamp(current["timestamp"])
            gap_seconds = (current_dt - previous_dt).total_seconds()

            if gap_seconds >= 0:
                gaps[current["sender"]].append(gap_seconds)

    averages = {}
    for person in participants:
        if gaps[person]:
            averages[person] = sum(gaps[person]) / len(gaps[person])
        else:
            averages[person] = None

    return averages, gaps

def longest_silent_streak(messages, participants, first_date, last_date):
    all_days = []
    current = first_date
    while current <= last_date:
        all_days.append(current)
        current += timedelta(days=1)

    active_by_person = {person: set() for person in participants}
    for msg in messages:
        dt = parse_timestamp(msg["timestamp"])
        active_by_person[msg["sender"]].add(dt.date())

    results = {}

    for person in participants:
        best = 0
        current_streak = 0
        best_start = None
        best_end = None
        streak_start = None

        for day in all_days:
            if day not in active_by_person[person]:
                if current_streak == 0:
                    streak_start = day
                current_streak += 1

                if current_streak > best:
                    best = current_streak
                    best_start = streak_start
                    best_end = day
            else:
                current_streak = 0
                streak_start = None

        results[person] = {
            "days": best,
            "start": best_start,
            "end": best_end
        }

    return results

def format_duration(seconds):
    if seconds is None:
        return "N/A"
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f} minutes"
    return f"{minutes / 60:.1f} hours"

if "people" in globals() and messages:
    avg_response, response_gap_lists = response_times(messages, people)

    first_date = min(parse_timestamp(m["timestamp"]).date() for m in messages)
    last_date = max(parse_timestamp(m["timestamp"]).date() for m in messages)
    silent = longest_silent_streak(messages, people, first_date, last_date)

    valid_response = [(p, v) for p, v in avg_response.items() if v is not None]
    fastest = min(valid_response, key=lambda x: x[1]) if valid_response else None
    slowest = max(valid_response, key=lambda x: x[1]) if valid_response else None

    print("RESPONSE PATTERNS")
    if fastest:
        print(f"Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
    if slowest:
        print(f"Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")

    print()
    print("LONGEST SILENT STREAKS")
    for person, data in sorted(silent.items(), key=lambda x: x[1]["days"], reverse=True):
        if data["days"] == 0:
            print(f"{person:<12}: 0 days")
        else:
            start = data["start"].strftime("%d %b")
            end = data["end"].strftime("%d %b")
            print(f"{person:<12}: {data['days']} days ({start} - {end})")

RESPONSE PATTERNS
Fastest replier : Vikas (avg 34.9 minutes)
Slowest replier : Aman (avg 54.9 minutes)

LONGEST SILENT STREAKS
Vikas       : 11 days (23 Apr - 03 May)
Karan       : 0 days
Rahul       : 0 days
Aman        : 0 days
Neha        : 0 days
Priya       : 0 days


## Feature 7 — Personality Archetype Detection

Eight quantitative archetypes are supported:

1. The Spammer
2. The Group Mom
3. The Night Owl
4. The Storyteller
5. The Drama Queen
6. The Ghost
7. The Comedian
8. The Question Master

Assignment is exclusive: each participant receives the highest-scoring archetype. The score functions are deliberately transparent so they can be explained in a viva.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CARING_WORDS = [
    "okay", "safe", "eat", "sleep", "take care", "are you",
    "please", "reminder", "drink water", "don't forget"
]

FUN_WORDS = ["lol", "lmao", "haha", "rofl", "lmfao"]

def message_bursts(person_messages):
    ordered = sorted(person_messages, key=lambda x: parse_timestamp(x["timestamp"]))
    bursts = []
    current = 0
    previous_sender = None

    # A burst is a run of consecutive messages by the same person.
    for msg in ordered:
        if previous_sender == msg["sender"]:
            current += 1
        else:
            if current:
                bursts.append(current)
            current = 1
        previous_sender = msg["sender"]

    if current:
        bursts.append(current)

    return bursts

def score_spammer(person_messages):
    bursts = message_bursts(person_messages)
    return sum(bursts) / len(bursts) if bursts else 0

def score_group_mom(person_messages):
    text_blob = " ".join(m["text"].lower() for m in person_messages)
    score = 0
    for phrase in CARING_WORDS:
        score += text_blob.count(phrase)
    return score / max(1, len(person_messages))

def score_night_owl(person_messages):
    if not person_messages:
        return 0
    night_count = 0
    for msg in person_messages:
        hour = parse_timestamp(msg["timestamp"]).hour
        if hour >= 23 or hour <= 4:
            night_count += 1
    return night_count / len(person_messages)

def score_storyteller(person_messages):
    if not person_messages:
        return 0
    total_words = sum(len(m["text"].split()) for m in person_messages)
    return total_words / len(person_messages)

def score_drama_queen(person_messages):
    if not person_messages:
        return 0
    dramatic = 0
    for msg in person_messages:
        text = msg["text"].strip()
        if len(text) >= 3 and text.isupper():
            dramatic += 1
        elif text.count("!") >= 2:
            dramatic += 1
    return dramatic / len(person_messages)

def score_ghost(person_messages, first_date, last_date):
    total_days = (last_date - first_date).days + 1
    active_days = set(parse_timestamp(m["timestamp"]).date() for m in person_messages)
    return 1 - (len(active_days) / total_days)

def score_comedian(person_messages):
    if not person_messages:
        return 0
    hits = 0
    for msg in person_messages:
        lower = msg["text"].lower()
        for word in FUN_WORDS:
            hits += lower.count(word)
    return hits / len(person_messages)

def score_question_master(person_messages):
    if not person_messages:
        return 0
    questions = sum(1 for m in person_messages if m["text"].strip().endswith("?"))
    return questions / len(person_messages)

def normalize_score(value, threshold):
    # Scores above the rule threshold receive proportionally stronger scores.
    if threshold <= 0:
        return value
    return value / threshold

def detect_archetypes(messages, participants):
    by_person = {person: [] for person in participants}
    for msg in messages:
        by_person[msg["sender"]].append(msg)

    first_date = min(parse_timestamp(m["timestamp"]).date() for m in messages)
    last_date = max(parse_timestamp(m["timestamp"]).date() for m in messages)

    raw = {}
    scored = {}

    for person in participants:
        pm = by_person[person]

        raw[person] = {
            "THE SPAMMER": score_spammer(pm),
            "THE GROUP MOM": score_group_mom(pm),
            "THE NIGHT OWL": score_night_owl(pm),
            "THE STORYTELLER": score_storyteller(pm),
            "THE DRAMA QUEEN": score_drama_queen(pm),
            "THE GHOST": score_ghost(pm, first_date, last_date),
            "THE COMEDIAN": score_comedian(pm),
            "THE QUESTION MASTER": score_question_master(pm)
        }

        thresholds = {
            "THE SPAMMER": 3,
            "THE GROUP MOM": 0.01,
            "THE NIGHT OWL": 0.60,
            "THE STORYTELLER": 30,
            "THE DRAMA QUEEN": 0.30,
            "THE GHOST": 0.60,
            "THE COMEDIAN": 0.01,
            "THE QUESTION MASTER": 0.25
        }

        scored[person] = {}
        for archetype, value in raw[person].items():
            scored[person][archetype] = normalize_score(value, thresholds[archetype])

    # Dataset-specific archetype selection is not hard-coded.
    # We rank normalized quantitative scores and assign exactly one label.
    assigned = {}
    runner_up = {}

    for person in participants:
        ranked = sorted(
            scored[person].items(),
            key=lambda item: item[1],
            reverse=True
        )
        assigned[person] = ranked[0][0]
        runner_up[person] = ranked[1][0]

    return assigned, runner_up, raw, scored

if "messages" in globals() and messages:
    archetypes, runners_up, archetype_raw, archetype_scores = detect_archetypes(messages, people)

    print("PERSONALITY ARCHETYPES")
    for person in sorted(archetypes):
        print(f"{person:<12} → {archetypes[person]:<20} | runner-up: {runners_up[person]}")

PERSONALITY ARCHETYPES
Aman         → THE SPAMMER          | runner-up: THE GROUP MOM
Karan        → THE SPAMMER          | runner-up: THE GROUP MOM
Neha         → THE SPAMMER          | runner-up: THE GROUP MOM
Priya        → THE SPAMMER          | runner-up: THE GROUP MOM
Rahul        → THE SPAMMER          | runner-up: THE COMEDIAN
Vikas        → THE COMEDIAN         | runner-up: THE SPAMMER


## Feature 8 — Final Report

The final report combines the core analytics into one terminal-style output. It is designed to be screenshot-friendly without using plotting libraries.

In [ ]:
def generate_final_report(messages, parsed, people, ranked_people, activity_matrix,
                          group_words, archetypes, silent, avg_response,
                          busiest_day, busiest_day_count, busiest_hour):
    first_dt = min(parse_timestamp(m["timestamp"]) for m in messages)
    last_dt = max(parse_timestamp(m["timestamp"]) for m in messages)
    period_days = (last_dt.date() - first_dt.date()).days + 1

    print()
    print("=" * 72)
    print('GROUPDNA REPORT — "Hostel Bois 4ever"')
    print(f"{period_days} days • {len(messages):,} analyzable messages • {len(people)} members")
    print("=" * 72)

    print(f"Period       : {first_dt.strftime('%d %B %Y')} to {last_dt.strftime('%d %B %Y')}")
    print(f"Busiest day  : {busiest_day.strftime('%d %B %Y')} ({busiest_day_count} messages)")
    print(f"Busiest hour : {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00")

    print()
    print("MESSAGES PER PERSON")
    for person, count in ranked_people:
        pct = count / len(messages) * 100
        bar_len = max(1, round(pct / 5))
        print(f"{person:<12} {'█' * bar_len:<20} {count:>5,} ({pct:>5.1f}%)")

    print()
    print("ACTIVITY HEATMAP (hour of day)")
    print("       " + " ".join(f"{h:02d}" for h in range(24)))
    for i, person in enumerate([p for p, _ in ranked_people]):
        row_max = int(np.max(activity_matrix[i]))
        symbols = " ".join(heat_symbol(int(v), row_max).strip() for v in activity_matrix[i])
        print(f"{person:<8}{symbols}")

    print()
    print("THIS GROUP'S FAVOURITE WORDS")
    ranked_words = sorted(group_words.items(), key=lambda x: x[1], reverse=True)[:10]
    if ranked_words:
        highest = ranked_words[0][1]
        for word, count in ranked_words:
            bar_len = max(1, round(count / highest * 20))
            print(f"{word:<15} {'█' * bar_len:<20} {count}")

    print()
    print("RESPONSE PATTERNS")
    valid = [(p, v) for p, v in avg_response.items() if v is not None]
    if valid:
        fastest = min(valid, key=lambda x: x[1])
        slowest = max(valid, key=lambda x: x[1])
        print(f"Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
        print(f"Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")

    print()
    print("LONGEST SILENT STREAKS")
    for person, data in sorted(silent.items(), key=lambda x: x[1]["days"], reverse=True):
        if data["days"]:
            print(f"{person:<12}: {data['days']} days ({data['start'].strftime('%d %b')} - {data['end'].strftime('%d %b')})")
        else:
            print(f"{person:<12}: 0 days")

    print()
    print("SPECIAL MESSAGE COUNTS")
    print(f"System messages : {parsed['system_count']}")
    print(f"Media entries   : {sum(parsed['media_count'].values())}")
    print(f"Deleted entries : {sum(parsed['deleted_count'].values())}")

    print()
    print("PERSONALITY ARCHETYPES")
    for person, _ in ranked_people:
        print(f"{person:<12} → {archetypes[person]}")

    print("=" * 72)
    print("Generated by GroupDNA • Built with Python + NumPy")
    print("=" * 72)

if "messages" in globals() and messages:
    avg_response, _ = response_times(messages, people)
    silent = longest_silent_streak(
        messages,
        people,
        min(parse_timestamp(m["timestamp"]).date() for m in messages),
        max(parse_timestamp(m["timestamp"]).date() for m in messages)
    )
    busiest_day, busiest_day_count, busiest_hour, _, _, _ = busiest_day_and_hour(messages)

    participant_order = [p for p, _ in ranked_people]
    activity_matrix = build_activity_matrix(messages, participant_order)
    group_words = word_frequencies(messages)
    archetypes, runners_up, archetype_raw, archetype_scores = detect_archetypes(messages, people)

    generate_final_report(
        messages, parsed, people, ranked_people, activity_matrix,
        group_words, archetypes, silent, avg_response,
        busiest_day, busiest_day_count, busiest_hour
    )


GROUPDNA REPORT — "Hostel Bois 4ever"
60 days • 3,127 analyzable messages • 6 members
Period       : 01 April 2024 to 30 May 2024
Busiest day  : 04 May 2024 (74 messages)
Busiest hour : 18:00 - 19:00

MESSAGES PER PERSON
Rahul        ██████                 940 ( 30.1%)
Priya        █████                  712 ( 22.8%)
Neha         ████                   624 ( 20.0%)
Aman         ███                    484 ( 15.5%)
Karan        ██                     345 ( 11.0%)
Vikas        █                       22 (  0.7%)

ACTIVITY HEATMAP (hour of day)
       00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul   ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █
Priya   . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░
Neha    . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒
Aman    █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █
Karan   . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░
Vikas   . . . . . . . ▒ █ ▒ ▒ . ▒ █ . ▒ ▒ █ █ █ ▒ ▒ ▒ █

THIS GROUP'S FAVOURITE WORDS
how    